# 0 - Téléchargement des données

Ce notebook illustre les différentes fonctionnalités des clients du répertoire pour télécharger des données via l'API SDMX. Chaque client permet de télécharger des données depuis une source distincte (OCDE etc ...)

## Table des matières

0. [Importation des modules](#section-0)
1. [Téléchargement des données de l'OCDE](#section-1)
    - 1.1 [Enumération des dataflows](#section-1.1)
    - 1.2 [Description de la structure d'un dataflow](#section-1.2)
    - 1.3 [Requête basique avec positions](#section-1.3)
    - 1.4 [Requête avec noms de dimensions](#section-1.4)
    - 1.5 [Formats de réponse](#section-1.5)
    - 1.6 [Séparation des requêtes avec split dimensions](#section-1.5)
    - 1.7 [Utilisation de OECDQueryRequest](#section-1.6)
    - 1.8 [Filtre par date de mise à jour](#section-1.7)
    - 1.9 [Chargement depuis un fichier de configuration](#section-1.8)
    - 1.10 [Gestion des erreurs](#section-1.9)
    - 1.11 [Memento et conseils de performance](#section-1.11)

## 0 - Importation des modules <a id="section-0"></a>

Importation des modules nécessaires et configuration de l'environnement.

In [1]:
# Rechargement automatique des modules
%load_ext autoreload
%autoreload 2

# Modules de base
import sys
import yaml
import pandas as pd
from pathlib import Path

# Ajout du répertoire parent au path
sys.path.append('..')

# Modules du package
from macroforecast.datasets.core import ResponseFormat
from macroforecast.datasets.sources import OECDClient, OECDQueryRequest

## 1 - Téléchargement des données de l'OCDE <a id="section-1"></a>

In [2]:
# Initialisation du client
client = OECDClient()

### 1.1 - Enumération des dataflows <a id="section-1.1"></a>

La méthode `list_all_dataflows()` permet d'énumérer tous les dataflows disponibles pour toute les agencies d'une source de données.

In [3]:
# Récupération de tous les dataflows
dataflows = client.list_all_dataflows()

# Affichage
print(f"Nombre total de dataflows: {len(dataflows)}")
print("\nPremiers dataflows:")
dataflows.head(10)

Nombre total de dataflows: 1481

Premiers dataflows:


,dataflow,agency,version,name
0,SEEA_AEA_A,ESTAT,1.4,Air Emissions Accounts
1,DF_SDG_GLC,IAEG-SDGs,1.20,SDG Country Global Dataflow
2,DF_SDG_GLH,IAEG-SDGs,1.20,SDG Harmonized Global Dataflow
3,DSD_FUA_CLIM@DF_CLIM_PROJ,OECD.CFE.EDS,1.4,"Climate projections by scenario, 2030–2060 – C..."
4,DSD_FUA_CLIM@DF_COASTAL_FLOOD,OECD.CFE.EDS,1.1,Coastal flooding - Cities and FUAs
5,DSD_FUA_CLIM@DF_DROUGHT,OECD.CFE.EDS,1.2,Drought - Cities and FUAs
6,DSD_FUA_CLIM@DF_FIRES,OECD.CFE.EDS,1.1,Wildfires - Cities and FUAs
7,DSD_FUA_CLIM@DF_HEAT_STRESS,OECD.CFE.EDS,1.1,Heat stress - Cities and FUAs
8,DSD_FUA_CLIM@DF_LAND_TEMP,OECD.CFE.EDS,1.2,Land surface temperature - Cities and FUAs
9,DSD_FUA_CLIM@DF_PRECIP,OECD.CFE.EDS,1.1,Precipitation - FUAs


### 1.2 - Description de la structure d'un dataflow <a id="section-1.2"></a>

La méthode `get_structure()` pour d'extraire les dimensions d'un dataflow.

In [4]:
# Extraction de la structure du dataflow KEI
structure = client.get_structure(
    agency="OECD.SDD.STES",
    dataflow="DSD_KEI@DF_KEI",
    version="4.0"
)

# Affichage
print(f"Agency: {structure.agency}")
print(f"Dataflow: {structure.dataflow}")
print(f"Nombre de dimensions: {structure.num_dimensions}")
print("\nDimensions:")
for dim in structure.dimensions:
    print(f"  Position {dim.position}: {dim.name} - {dim.description}")

Agency: OECD.SDD.STES
Dataflow: DSD_KEI@DF_KEI
Nombre de dimensions: 7

Dimensions:
  Position 0: REF_AREA - Reference area
  Position 1: FREQ - Frequency of observation
  Position 2: MEASURE - Measure
  Position 3: UNIT_MEASURE - Unit of measure
  Position 4: ACTIVITY - Economic activity
  Position 5: ADJUSTMENT - Adjustment
  Position 6: TRANSFORMATION - Transformation


### 1.3 - Requête basique avec positions <a id="section-1.3"></a>

La méthode `get_data()` permet de requêter les données d'un dataflow en appliquant des filtres sur les dimensions désignées par leur position (0, 1, 2...) dans l'URL de requête.

In [5]:
# Requête avec positions numériques
df = client.get_data(
    agency="OECD.SDD.STES",
    dataflow="DSD_KEI@DF_KEI",
    version="+",
    dimensions={
        0: ["FRA"],  # Position 0 = REF_AREA
        1: ["M"],    # Position 1 = FREQ
        2: ["LI"],   # Position 2 = MEASURE
    },
    start_period="2020"
)

# Affichage
print(f"Nombre de lignes: {len(df)}")
print(f"Période: {df['TIME_PERIOD'].min()} - {df['TIME_PERIOD'].max()}")
df.head()

Nombre de lignes: 72
Période: 2020-01 - 2025-12


,STRUCTURE,STRUCTURE_ID,STRUCTURE_NAME,ACTION,REF_AREA,Reference area,FREQ,Frequency of observation,MEASURE,Measure,...,OBS_VALUE,Observation value,OBS_STATUS,Observation status,UNIT_MULT,Unit multiplier,DECIMALS,Decimals,BASE_PER,Base period
0,DATAFLOW,OECD.SDD.STES:DSD_KEI@DF_KEI(4.0),Key short-term economic indicators,I,FRA,France,M,Monthly,LI,Composite leading indicator (CLI),...,99.192830,NaN,A,Normal value,0,Units,1,One,NaN,NaN
1,DATAFLOW,OECD.SDD.STES:DSD_KEI@DF_KEI(4.0),Key short-term economic indicators,I,FRA,France,M,Monthly,LI,Composite leading indicator (CLI),...,99.167547,NaN,A,Normal value,0,Units,1,One,NaN,NaN
2,DATAFLOW,OECD.SDD.STES:DSD_KEI@DF_KEI(4.0),Key short-term economic indicators,I,FRA,France,M,Monthly,LI,Composite leading indicator (CLI),...,99.198761,NaN,A,Normal value,0,Units,1,One,NaN,NaN
3,DATAFLOW,OECD.SDD.STES:DSD_KEI@DF_KEI(4.0),Key short-term economic indicators,I,FRA,France,M,Monthly,LI,Composite leading indicator (CLI),...,98.428563,NaN,A,Normal value,0,Units,1,One,NaN,NaN
4,DATAFLOW,OECD.SDD.STES:DSD_KEI@DF_KEI(4.0),Key short-term economic indicators,I,FRA,France,M,Monthly,LI,Composite leading indicator (CLI),...,99.798374,NaN,A,Normal value,0,Units,1,One,NaN,NaN


### 1.4 - Requête avec noms de dimensions <a id="section-1.4"></a>

La méthode `get_data()` permet également de requêter les données d'un dataflow en appliquant des filtres sur les dimensions désignées par leur nom (plus lisible et robuste).

In [6]:
# Requête avec noms de dimensions
df = client.get_data(
    agency="OECD.SDD.STES",
    dataflow="DSD_KEI@DF_KEI",
    version="+",
    dimensions={
        "REF_AREA": ["FRA", "DEU"],  # France et Allemagne
        "FREQ": "M",                  # Fréquence mensuelle
        "MEASURE": "LI",              # Leading Indicator
    },
    start_period="2020"
)

# Affichage
print(f"Nombre de lignes: {len(df)}")
print(f"Pays: {sorted(df['REF_AREA'].unique())}")
df.head()

Nombre de lignes: 144
Pays: ['DEU', 'FRA']


,STRUCTURE,STRUCTURE_ID,STRUCTURE_NAME,ACTION,REF_AREA,Reference area,FREQ,Frequency of observation,MEASURE,Measure,...,OBS_VALUE,Observation value,OBS_STATUS,Observation status,UNIT_MULT,Unit multiplier,DECIMALS,Decimals,BASE_PER,Base period
120,DATAFLOW,OECD.SDD.STES:DSD_KEI@DF_KEI(4.0),Key short-term economic indicators,I,FRA,France,M,Monthly,LI,Composite leading indicator (CLI),...,100.860252,NaN,A,Normal value,0,Units,1,One,NaN,NaN
121,DATAFLOW,OECD.SDD.STES:DSD_KEI@DF_KEI(4.0),Key short-term economic indicators,I,FRA,France,M,Monthly,LI,Composite leading indicator (CLI),...,100.351917,NaN,A,Normal value,0,Units,1,One,NaN,NaN
122,DATAFLOW,OECD.SDD.STES:DSD_KEI@DF_KEI(4.0),Key short-term economic indicators,I,FRA,France,M,Monthly,LI,Composite leading indicator (CLI),...,99.834894,NaN,A,Normal value,0,Units,1,One,NaN,NaN
123,DATAFLOW,OECD.SDD.STES:DSD_KEI@DF_KEI(4.0),Key short-term economic indicators,I,FRA,France,M,Monthly,LI,Composite leading indicator (CLI),...,99.114891,NaN,A,Normal value,0,Units,1,One,NaN,NaN
124,DATAFLOW,OECD.SDD.STES:DSD_KEI@DF_KEI(4.0),Key short-term economic indicators,I,FRA,France,M,Monthly,LI,Composite leading indicator (CLI),...,99.210289,NaN,A,Normal value,0,Units,1,One,NaN,NaN


### 1.5 - Formats de réponse <a id="section-1.5"></a>

L'API SDMX de l'OCDE supporte plusieurs formats de réponse. Le client permet de spécifier le format souhaité via le paramètre `format` de `OECDQueryRequest`.

In [7]:
# Affichage des formats disponibles
print("Formats de réponse disponibles:")
for fmt in ResponseFormat:
    print(f"  - {fmt.name}: {fmt.value}")

Formats de réponse disponibles:
  - JSON: json
  - CSV: csv
  - CSV_LABELS: csv_labels
  - XML: xml


In [8]:
# Comparaison des formats CSV et CSV_LABELS
query_csv = OECDQueryRequest(
    agency="OECD.SDD.STES",
    dataflow="DSD_KEI@DF_KEI",
    version="+",
    dimensions={
        "REF_AREA": ["FRA"],
        "FREQ": "M",
        "MEASURE": "LI"
    },
    start_period="2024",
    format=ResponseFormat.CSV  # Format avec codes uniquement
)

query_csv_labels = OECDQueryRequest(
    agency="OECD.SDD.STES",
    dataflow="DSD_KEI@DF_KEI",
    version="+",
    dimensions={
        "REF_AREA": ["FRA"],
        "FREQ": "M",
        "MEASURE": "LI"
    },
    start_period="2024",
    format=ResponseFormat.CSV_LABELS  # Format avec labels lisibles (défaut)
)

# Exécution des requêtes
df_csv = client.execute_query(query_csv)
df_csv_labels = client.execute_query(query_csv_labels)

# Comparaison des colonnes
print("Colonnes avec CSV:")
print(f"  {list(df_csv.columns)}")
print("\nColonnes avec CSV_LABELS:")
print(f"  {list(df_csv_labels.columns)}")

# Aperçu des données
print("\n--- Format CSV (codes uniquement) ---")
print(df_csv[['REF_AREA', 'MEASURE', 'TIME_PERIOD', 'OBS_VALUE']].head(3).to_string(index=False))

print("\n--- Format CSV_LABELS (avec labels) ---")
# Affichage des colonnes de labels si présentes
label_cols = [c for c in df_csv_labels.columns if 'Reference area' in c or 'Measure' in c]
if label_cols:
    display_cols = ['REF_AREA'] + label_cols[:1] + ['MEASURE', 'TIME_PERIOD', 'OBS_VALUE']
    display_cols = [c for c in display_cols if c in df_csv_labels.columns]
    print(df_csv_labels[display_cols].head(3).to_string(index=False))

Colonnes avec CSV:
  ['STRUCTURE', 'STRUCTURE_ID', 'ACTION', 'REF_AREA', 'FREQ', 'MEASURE', 'UNIT_MEASURE', 'ACTIVITY', 'ADJUSTMENT', 'TRANSFORMATION', 'TIME_PERIOD', 'OBS_VALUE', 'OBS_STATUS', 'UNIT_MULT', 'DECIMALS', 'BASE_PER']

Colonnes avec CSV_LABELS:
  ['STRUCTURE', 'STRUCTURE_ID', 'STRUCTURE_NAME', 'ACTION', 'REF_AREA', 'Reference area', 'FREQ', 'Frequency of observation', 'MEASURE', 'Measure', 'UNIT_MEASURE', 'Unit of measure', 'ACTIVITY', 'Economic activity', 'ADJUSTMENT', 'Adjustment', 'TRANSFORMATION', 'Transformation', 'TIME_PERIOD', 'Time period', 'OBS_VALUE', 'Observation value', 'OBS_STATUS', 'Observation status', 'UNIT_MULT', 'Unit multiplier', 'DECIMALS', 'Decimals', 'BASE_PER', 'Base period']

--- Format CSV (codes uniquement) ---
REF_AREA MEASURE TIME_PERIOD  OBS_VALUE
     FRA      LI     2024-04  99.109699
     FRA      LI     2024-03  99.167547
     FRA      LI     2025-12 101.314910

--- Format CSV_LABELS (avec labels) ---
REF_AREA Reference area MEASURE TIME_PE

### 1.6 - Séparation des requêtes avec split dimensions <a id="section-1.6"></a>

Le paramètre `split_dimensions` de la méthode `get_data()` permet d'effectuer une requête distincte pour chaque valeur de la dimension spécifiée. Cela permet de gérer les requêtes volumineuses.

In [9]:
# Requête avec split_dimensions
# Au lieu d'une seule requête pour 7 pays, on génère 7 requêtes séparées
df = client.get_data(
    agency="OECD.SDD.STES",
    dataflow="DSD_KEI@DF_KEI",
    version="+",
    dimensions={
        "REF_AREA": ["CAN", "FRA", "DEU", "ITA", "JPN", "GBR", "USA"],
        "FREQ": "M",
        "MEASURE": "LI",
    },
    start_period="2020",
    split_dimensions=["REF_AREA"]  # Générera 7 requêtes séparées
)

# Affichage
print(f"Nombre de lignes: {len(df)}")
print(f"Pays: {sorted(df['REF_AREA'].unique())}")
print("\nStatistiques par pays:")
print(df.groupby('REF_AREA').size())

Nombre de lignes: 504
Pays: ['CAN', 'DEU', 'FRA', 'GBR', 'ITA', 'JPN', 'USA']

Statistiques par pays:
REF_AREA
CAN    72
DEU    72
FRA    72
GBR    72
ITA    72
JPN    72
USA    72
dtype: int64


### 1.7 - Utilisation de OECDQueryRequest <a id="section-1.7"></a>

L'utilisation de l'objet `QueryRequest` permet la création et l'exécution d'une requête en manipulant ses paramètres de manière plus flexible qu'à travers l'utilisation des arguments de `get_data`.

In [10]:
# Création d'une QueryRequest
query = OECDQueryRequest(
    agency="OECD.SDD.STES",
    dataflow="DSD_KEI@DF_KEI",
    version="+",
    dimensions={
        "REF_AREA": ["FRA"],
        "FREQ": "M",
        "MEASURE": "LI"
    },
    start_period="2020"
)

# Affichage
print(f"Dataflow key: {query.get_dataflow_key()}")
print(f"Dimensions: {query.dimensions}")

# Exécution de la requête
df = client.execute_query(query)

# Affichage
print(f"\nNombre de lignes: {len(df)}")
print(f"Période: {df['TIME_PERIOD'].min()} - {df['TIME_PERIOD'].max()}")
df.head()

Dataflow key: OECD.SDD.STES::DSD_KEI@DF_KEI::+
Dimensions: {'REF_AREA': ['FRA'], 'FREQ': 'M', 'MEASURE': 'LI'}

Nombre de lignes: 72
Période: 2020-01 - 2025-12


,STRUCTURE,STRUCTURE_ID,STRUCTURE_NAME,ACTION,REF_AREA,Reference area,FREQ,Frequency of observation,MEASURE,Measure,...,OBS_VALUE,Observation value,OBS_STATUS,Observation status,UNIT_MULT,Unit multiplier,DECIMALS,Decimals,BASE_PER,Base period
0,DATAFLOW,OECD.SDD.STES:DSD_KEI@DF_KEI(4.0),Key short-term economic indicators,I,FRA,France,M,Monthly,LI,Composite leading indicator (CLI),...,99.192830,NaN,A,Normal value,0,Units,1,One,NaN,NaN
1,DATAFLOW,OECD.SDD.STES:DSD_KEI@DF_KEI(4.0),Key short-term economic indicators,I,FRA,France,M,Monthly,LI,Composite leading indicator (CLI),...,99.167547,NaN,A,Normal value,0,Units,1,One,NaN,NaN
2,DATAFLOW,OECD.SDD.STES:DSD_KEI@DF_KEI(4.0),Key short-term economic indicators,I,FRA,France,M,Monthly,LI,Composite leading indicator (CLI),...,99.198761,NaN,A,Normal value,0,Units,1,One,NaN,NaN
3,DATAFLOW,OECD.SDD.STES:DSD_KEI@DF_KEI(4.0),Key short-term economic indicators,I,FRA,France,M,Monthly,LI,Composite leading indicator (CLI),...,98.428563,NaN,A,Normal value,0,Units,1,One,NaN,NaN
4,DATAFLOW,OECD.SDD.STES:DSD_KEI@DF_KEI(4.0),Key short-term economic indicators,I,FRA,France,M,Monthly,LI,Composite leading indicator (CLI),...,99.798374,NaN,A,Normal value,0,Units,1,One,NaN,NaN


### 1.8 - Filtre par date de mise à jour <a id="section-1.8"></a>

La méthode `filter_updated_queries()` permet d'éviter de télécharger des données qui n'ont pas été mises à jour antérieurement à une certaine date.

In [11]:
# Création de plusieurs requêtes
queries = [
    OECDQueryRequest(
        agency="OECD.SDD.STES",
        dataflow="DSD_KEI@DF_KEI",
        dimensions={"REF_AREA": ["FRA"], "FREQ": "M"},
        start_period="2020"
    ),
    OECDQueryRequest(
        agency="OECD.SDD.STES",
        dataflow="DSD_KEI@DF_KEI",
        dimensions={"REF_AREA": ["DEU"], "FREQ": "M"},
        start_period="2020"
    ),
]

# Affichage
print(f"Nombre de requêtes totales: {len(queries)}")

# Test 1: Filtrage avec date ancienne (devrait tout garder)
print("\n--- Test avec date ancienne (2020-01-01) ---")
updated_queries_old = client.filter_updated_queries(
    queries,
    updated_since="2020-01-01"
)
print(f"Requêtes filtrées: {len(updated_queries_old)}/{len(queries)}")

# Test 2: Filtrage avec date récente
print("\n--- Test avec date récente (2024-01-01) ---")
updated_queries_recent = client.filter_updated_queries(
    queries,
    updated_since="2024-01-01"
)
print(f"Requêtes filtrées: {len(updated_queries_recent)}/{len(queries)}")

# Test 3: Aucun filtrage avec None
print("\n--- Test sans filtrage (updated_since=None) ---")
all_queries = client.filter_updated_queries(
    queries,
    updated_since=None
)
print(f"Requêtes retournées: {len(all_queries)}/{len(queries)}")

HTTP error: 406 Client Error: Not Acceptable for url: https://sdmx.oecd.org/public/rest/contentconstraint/OECD.SDD.STES/CR_A_DF_KEI
Response content: acceptable: application/vnd.sdmx.structure+xml; charset=utf-8; version=2.1, application/vnd.sdmx.structure+json; charset=utf-8; version=1.0, application/vnd.sdmx.structure+json; charset=utf-8; version=1.0.0-wd, text/json; charset=utf-8, application/json; charset=utf-8, application/rdf+xml; charset=utf-8, application/vnd.sdmx.structure+xml; charset=utf-8; version=2.0, application/xml; charset=utf-8; version=2.1, text/xml; charset=utf-8; version=2.1
optional : urn=true


Nombre de requêtes totales: 2

--- Test avec date ancienne (2020-01-01) ---
Requêtes filtrées: 0/2

--- Test avec date récente (2024-01-01) ---


HTTP error: 406 Client Error: Not Acceptable for url: https://sdmx.oecd.org/public/rest/contentconstraint/OECD.SDD.STES/CR_A_DF_KEI
Response content: acceptable: application/vnd.sdmx.structure+xml; charset=utf-8; version=2.1, application/vnd.sdmx.structure+json; charset=utf-8; version=1.0, application/vnd.sdmx.structure+json; charset=utf-8; version=1.0.0-wd, text/json; charset=utf-8, application/json; charset=utf-8, application/rdf+xml; charset=utf-8, application/vnd.sdmx.structure+xml; charset=utf-8; version=2.0, application/xml; charset=utf-8; version=2.1, text/xml; charset=utf-8; version=2.1
optional : urn=true


Requêtes filtrées: 0/2

--- Test sans filtrage (updated_since=None) ---
Requêtes retournées: 2/2


### 1.9 - Chargement des requêtes depuis un fichier de configuration <a id="section-1.9"></a>

Les requêtes à effectuer peuvent également êre instanciées depuis un fichier YAML de configuration de la manière suivante.

In [12]:
# Chargement de la configuration
config_path = Path('../config/datasets/oecd.yaml')
with open(config_path, 'r', encoding='utf-8') as f:
    config = yaml.safe_load(f)

print("Requêtes configurées:")
if 'queries' in config:
    # Enumération des requêtes disponibles
    for query_name in config['queries'].keys():
        print(f"  - {query_name}")

    # Exemple: chargement et exécution de la première requête
    query_cfg = config['queries'][list(config['queries'].keys())[0]]

    # Affichage des caractéristiques de la requête
    print(f"\nConfiguration de 'kei_g7_monthly':")
    print(f"  Agency: {query_cfg['agency']}")
    print(f"  Dataflow: {query_cfg['dataflow']}")
    print(f"  Dimensions: {list(query_cfg['dimensions'].keys())}")

    # Création de la QueryRequest depuis la config
    query = OECDQueryRequest(
        agency=query_cfg['agency'],
        dataflow=query_cfg['dataflow'],
        version=query_cfg.get('version', '+'),
        dimensions=query_cfg['dimensions'],
        start_period=query_cfg.get('start_period'),
        end_period=query_cfg.get('end_period'),
        split_dimensions=query_cfg.get('split_dimensions')
    )

    # Affichage
    print(f"\nQueryRequest créée: {query.get_dataflow_key()}")
    print("\nPour exécuter la requête, décommentez la ligne suivante:")
    print("# df = client.execute_query(query)")

else:
    print("⚠ Section 'query_requests' non trouvée dans la configuration")

Requêtes configurées:
  - kei_g7_monthly
  - nad_g7_quarterly

Configuration de 'kei_g7_monthly':
  Agency: OECD.SDD.STES
  Dataflow: DSD_KEI@DF_KEI
  Dimensions: ['REF_AREA', 'FREQ', 'UNIT_MEASURE', 'ACTIVITY', 'ADJUSTMENT', 'TRANSFORMATION']

QueryRequest créée: OECD.SDD.STES::DSD_KEI@DF_KEI::+

Pour exécuter la requête, décommentez la ligne suivante:
# df = client.execute_query(query)


### 1.10 - Gestion des erreurs <a id="section-1.10"></a>

Démonstration de la gestion des erreurs avec des requêtes invalides.

In [13]:
# Test 1: Agency invalide
print("Test 1: Agency invalide")
try:
    df = client.get_data(
        agency="INVALID_AGENCY",
        dataflow="INVALID_DATAFLOW",
        dimensions={"REF_AREA": ["FRA"]}
    )
except Exception as e:
    print(f"✓ Erreur attendue: {type(e).__name__}")
    print(f"  Message: {str(e)[:100]}")

# Test 2: Dimension invalide
print("\nTest 2: Dimension invalide")
try:
    df = client.get_data(
        agency="OECD.SDD.STES",
        dataflow="DSD_KEI@DF_KEI",
        dimensions={"INVALID_DIM": ["VALUE"]}
    )
except Exception as e:
    print(f"✓ Erreur attendue: {type(e).__name__}")
    print(f"  Message: {str(e)[:100]}")

print("\n✓ Tests de gestion d'erreurs terminés")

Test 1: Agency invalide


HTTP error: 404 Client Error: Not Found for url: https://sdmx.oecd.org/public/rest/v2/structure/dataflow/INVALID_AGENCY/INVALID_DATAFLOW/+?references=all&detail=referencepartial
Response content: Could not find requested structures
Failed to fetch structure for INVALID_AGENCY::INVALID_DATAFLOW: 404 Client Error: Not Found for url: https://sdmx.oecd.org/public/rest/v2/structure/dataflow/INVALID_AGENCY/INVALID_DATAFLOW/+?references=all&detail=referencepartial


✓ Erreur attendue: ValueError
  Message: Structure not found for INVALID_AGENCY::INVALID_DATAFLOW. Unable to resolve dimension name 'REF_AREA

Test 2: Dimension invalide
✓ Erreur attendue: ValueError
  Message: Dimension 'INVALID_DIM' not found in OECD.SDD.STES::DSD_KEI@DF_KEI. Available dimensions: ['REF_AREA

✓ Tests de gestion d'erreurs terminés


### 1.11: Memento et conseils de performance <a id="section-10"></a>

Meilleures pratiques pour optimiser les téléchargements de données OECD.

### 1. Rate Limiting
- Le client OECD implémente automatiquement un [rate limiting](https://www.oecd.org/fr/data/insights/data-explainers/2024/11/Api-best-practices-and-recommendations.html) de **60 requêtes par heure**
- Aucune action requise, c'est géré automatiquement

### 2. Split Dimensions
- Utiliser `split_dimensions` pour les requêtes avec beaucoup de valeurs
- Génère plusieurs petites requêtes au lieu d'une grosse requête
- Exemple: 7 pays × 14 transactions = 98 requêtes avec `split_dimensions=["REF_AREA", "TRANSACTION"]`

```python
# Séparation en plusieurs requêtes
df = client.get_data(
    agency="OECD.SDD.STES",
    dataflow="DSD_KEI@DF_KEI",
    dimensions={"REF_AREA": ["CAN", "FRA", "DEU", "ITA", "JPN", "GBR", "USA"]},
    split_dimensions=["REF_AREA"]
)
```

### 3. Filtrage par date
- Utiliser `filter_updated_queries()` pour éviter les téléchargements inutiles
- Ne télécharge que les dataflows mis à jour depuis une date donnée
- Passer `updated_since=None` pour ne pas filtrer

```python
# Ne télécharge que les données mises à jour cette semaine
updated_queries = client.filter_updated_queries(
    queries,
    updated_since="2024-12-15"
)
```

### 4. Formats de réponse
- **CSV_LABELS** (défaut): CSV avec labels lisibles - recommandé
- **CSV**: CSV avec codes - plus léger mais moins lisible
- **JSON**: Format JSON - plus flexible mais plus lourd

```python
from macroforecast.datasets.sdmx import ResponseFormat

query = QueryRequest(
    agency="OECD.SDD.STES",
    dataflow="DSD_KEI@DF_KEI",
    format=ResponseFormat.CSV  # Plus rapide si vous connaissez les codes
)
```

### 5. Gestion des doublons
- Par défaut: `on_duplicate="warn"` (affiche un warning)
- Options: `"ignore"`, `"warn"`, `"raise"`

```python
query = QueryRequest(
    agency="OECD.SDD.STES",
    dataflow="DSD_KEI@DF_KEI",
    on_duplicate="raise"  # Lève une exception en cas de doublon
)
```

### 6. Limitation temporelle
- Utiliser `start_period` et `end_period` pour limiter la période
- Ou `last_n_observations` pour ne récupérer que les N dernières observations

```python
# Seulement les 12 derniers mois
df = client.get_data(
    agency="OECD.SDD.STES",
    dataflow="DSD_KEI@DF_KEI",
    dimensions={"REF_AREA": ["FRA"]},
    last_n_observations=12
)
```

## Conclusion

Ce notebook a démontré toutes les fonctionnalités principales :

- du client OECD (énumération des dataflows, extraction de leur structure, exécution de requête avec différents paramètres ...) ; 

Pour plus d'informations, consulter:
- Le fichier `macroforecast/datasets/oecd.py`
- La configuration `config/datasets/oecd.yaml`